In [3]:
"""Run one observable edge-at-a-time K43 construction from scratch."""

from __future__ import annotations

from time import perf_counter

import numpy as np

from ramsey.RConstructionEdgeRisk import (
    REdgeRiskConstruction,
    REdgeRiskConstructionProgress,
)
from ramsey.RGraph import RGraph
from ramsey.RProblem import RProblem
from ramsey.RScoring import score_coloring


N_VERTICES = 43
RANDOM_SEED = 202_608_555
MINIMUM_PRESSURE_EDGES = 4
REPORT_EVERY = 100


def pressure_name(
    event: REdgeRiskConstructionProgress,
) -> str:
    """Describe whether an immediate monochromatic K5 is avoidable."""
    red_bad = event.red_completions > 0
    blue_bad = event.blue_completions > 0

    if red_bad and blue_bad:
        return "UNAVOIDABLE"

    if red_bad or blue_bad:
        return "ONE-SIDED"

    return "SAFE"


def main() -> None:
    """Construct one coloring, report progress, and verify the result."""
    graph_start = perf_counter()

    graph = RGraph(
        RProblem.r55(
            n_vertices=N_VERTICES,
        )
    )

    graph_elapsed = perf_counter() - graph_start

    safe_steps = 0
    one_sided_steps = 0
    unavoidable_steps = 0
    created_monochromatic_k5s = 0
    first_unavoidable_step: int | None = None

    def report_progress(
        event: REdgeRiskConstructionProgress,
    ) -> None:
        nonlocal safe_steps
        nonlocal one_sided_steps
        nonlocal unavoidable_steps
        nonlocal created_monochromatic_k5s
        nonlocal first_unavoidable_step

        pressure = pressure_name(event)

        if pressure == "SAFE":
            safe_steps += 1
        elif pressure == "ONE-SIDED":
            one_sided_steps += 1
        else:
            unavoidable_steps += 1

            if first_unavoidable_step is None:
                first_unavoidable_step = event.step_number

        created_now = event.created_monochromatic_k5s
        created_monochromatic_k5s += created_now

        should_report = (
            event.step_number == 1
            or event.step_number % REPORT_EVERY == 0
            or event.completed
            or (
                pressure == "UNAVOIDABLE"
                and event.step_number == first_unavoidable_step
            )
        )

        if not should_report:
            return

        color = "R" if event.color == 0 else "B"
        mode = "risk" if event.risk_driven else "alternate"

        print(
            f"Edge {event.step_number:3d}/{event.total_edges} | "
            f"edge={event.edge:3d} {event.endpoints!s:8s} | "
            f"chose={color} | "
            f"mode={mode:9s} | "
            f"R/B={event.red_edges:3d}/{event.blue_edges:3d} | "
            f"pressure={pressure:11s} | "
            f"complete-if-R/B="
            f"{event.red_completions}/{event.blue_completions} | "
            f"created={created_now:2d} | "
            f"total-mono={created_monochromatic_k5s:4d}"
        )

    construction = REdgeRiskConstruction(
        rng=np.random.default_rng(RANDOM_SEED),
        minimum_pressure_edges=MINIMUM_PRESSURE_EDGES,
        observer=report_progress,
    )

    print("Edge-Risk Construction")
    print("======================")
    print("Vertices:", graph.problem.n_vertices)
    print("Edges:", graph.number_of_edges)
    print("K5s:", graph.subgraph_index(5).clique_count)
    print("Seed:", RANDOM_SEED)
    print("Pressure threshold:", MINIMUM_PRESSURE_EDGES)
    print("Graph setup:", f"{graph_elapsed:.3f}s")
    print()

    construction_start = perf_counter()
    coloring = construction.construct(graph)
    construction_elapsed = perf_counter() - construction_start

    score_start = perf_counter()
    final_score = score_coloring(coloring)
    score_elapsed = perf_counter() - score_start

    report = construction.last_report

    if report is None:
        raise RuntimeError("Construction did not produce its final report.")

    if report.total_edges != graph.number_of_edges:
        raise RuntimeError(
            "Construction ended before every edge was colored."
        )

    if created_monochromatic_k5s != final_score:
        raise RuntimeError(
            "Incremental monochromatic count disagrees with exact score: "
            f"{created_monochromatic_k5s} versus {final_score}."
        )

    print()
    print("Final Result")
    print("============")
    print("Score:", final_score)
    print("Red edges:", report.red_edges)
    print("Blue edges:", report.blue_edges)
    print("Alternating decisions:", report.alternating_decisions)
    print("Risk decisions:", report.risk_decisions)
    print("Risk overrides:", report.risk_overrides)
    print("First risk step:", report.first_risk_step)
    print("Safe assignments:", safe_steps)
    print("One-sided assignments:", one_sided_steps)
    print("Unavoidable assignments:", unavoidable_steps)
    print("First unavoidable step:", first_unavoidable_step)
    print(
        "Incrementally created monochromatic K5s:",
        created_monochromatic_k5s,
    )
    print("Construction time:", f"{construction_elapsed:.3f}s")
    print("Exact scoring time:", f"{score_elapsed:.3f}s")
    print()
    print("All 903 edges were assigned: YES")
    print("Incremental count matches exact score: YES")


if __name__ == "__main__":
    main()

Edge-Risk Construction
Vertices: 43
Edges: 903
K5s: 962598
Seed: 202608555
Pressure threshold: 4
Graph setup: 1.455s

Edge   1/903 | edge=770 (26, 30) | chose=R | mode=alternate | R/B=  1/  0 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 | total-mono=   0
Edge 100/903 | edge=861 (33, 37) | chose=B | mode=risk      | R/B= 54/ 46 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 | total-mono=   0
Edge 200/903 | edge=144 (3, 25)  | chose=R | mode=risk      | R/B=101/ 99 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 | total-mono=   0
Edge 300/903 | edge=639 (19, 32) | chose=B | mode=risk      | R/B=151/149 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 | total-mono=   0
Edge 400/903 | edge=805 (28, 36) | chose=B | mode=risk      | R/B=199/201 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 | total-mono=   0
Edge 500/903 | edge=205 (5, 11)  | chose=R | mode=risk      | R/B=251/249 | pressure=SAFE        | complete-if-R/B=0/0 | created= 0 